# 02 · Baseline: M0 (правила) и M1 (TF-IDF + LogReg)
Порог каждой модели подбирается на val по условию FPR ≤ 2%.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO_DIR = '/content/prompt-injection-detector'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HPAINH/prompt-injection-detector.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['PIG_ROOT'] = '/content/drive/MyDrive/prompt-injection'
os.makedirs(os.environ['PIG_ROOT'], exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s', force=True)
from pig.config import load_config, set_seed
cfg = load_config()
set_seed(cfg['seed'])
print('PIG_ROOT =', os.environ['PIG_ROOT'])

In [ ]:
from pig.pipeline import load_data, calibrate
from pig.models.rules import RuleDetector
from pig.models.tfidf import TfidfDetector
from pig.eval.metrics import evaluate, summary_table
splits = load_data(cfg)
train, val, test = splits['train'], splits['val'], splits['test']
max_fpr = cfg['threshold']['max_fpr']
md_dir = cfg.path('models')

In [ ]:
m0 = RuleDetector()
calibrate(m0, val, max_fpr)
m0.save(md_dir / 'M0_rules.joblib')

tc = cfg['models']['tfidf']
m1 = TfidfDetector(tc['word_ngrams'], tc['char_ngrams'], tc['max_features'], tc['C'], cfg['seed'])
m1.fit(train['text'], train['label'])
calibrate(m1, val, max_fpr)
m1.save(md_dir / 'M1_tfidf.joblib')
print('пороги:', m0.threshold, m1.threshold)

In [ ]:
res = [evaluate(m.name, test, m.predict_proba(test['text'].tolist()), m.threshold) for m in [m0, m1]]
summary_table(res)

## M0 с порогом 0.5 (как правила работают «из коробки»)

In [ ]:
evaluate('M0_rules@0.5', test, m0.predict_proba(test['text'].tolist()), 0.5, ci=False)

## Самые весомые признаки M1

In [ ]:
tf = m1.top_features(20)
print('атака:', tf['attack'])
print('норма:', tf['benign'])